# 实验（非计分） - 追踪 RAG 系统

---

欢迎来到关于使用 Weaviate 和 Phoenix 追踪及评估 RAG 系统的非计分实验！在本次互动环节中，你将学习如何有效地利用遥测技术（telemetry）来追踪 RAG 系统并排除故障。你将理解并运用一些核心概念，包括 span、trace 和 chain，这些对于监控和提升系统性能至关重要。



在本实验中，你将：

- 了解如何设置和使用遥测技术来监控你的 RAG 系统。
- 学习 trace 和 span 的相关知识。
- 探索 trace，以查看系统流程中的完整路径和交互过程。
- 使用 [Phoenix Arize](https://phoenix.arize.com/) 工具对遥测数据进行可视化和分析。
- 观察结合了 Phoenix 和 Weaviate 的小型 RAG 流水线的实际运作。

---

<h4 style="color:black; font-weight:bold;">使用目录</h4>
JupyterLab 为你提供了一种在作业中进行导航的简便方法。它位于左侧面板的“Table of Contents”（目录）选项卡下，如下图所示。

![TOC Location](images/toc.png)

---

# 目录
- [ 1 - 简介](#1)
  - [ 1.1 导入必要的库](#1-1)
- [ 2 - 遥测技术简介](#2)
  - [ 2.1 Span (跨度)](#2-1)
- [ 3 - 使用 Phoenix 进行遥测](#3)
  - [ 3.1 启动 Phoenix 应用](#3-1)
  - [ 3.2 准备遥测环境](#3-2)
  - [ 3.3 运行流水线](#3-3)
  - [ 3.4 Chain (链)](#3-4)
  - [ 3.5 使用 UI 分析追踪数据](#3-5)
- [ 4 - 结合 Weaviate 的追踪与评估](#4)
  - [ 4.1 配置追踪器](#4-1)
  - [ 4.2 准备 Weaviate 客户端和集合](#4-2)
  - [ 4.3 检索器 (Retriever)](#4-3)
  - [ 4.4 使用 `openai` 库调用 LLM](#4-4)
- [ 5 - 评估 RAG 系统](#5)

<a id='1'></a>
## 1 - 简介

---
在 RAG 系统的背景下，遥测技术（telemetry）是监控和优化性能的关键。通过收集和传输系统运行的数据，例如 span（单个步骤）和 trace（完整工作流），遥测技术提供了一种观察系统如何检索、处理和生成信息的方法。这种可见性有助于识别瓶颈和诊断问题，从而提高系统效率。



<a id='1-1'></a>
### 1.1 导入必要的库

In [2]:
import utils  # 导入自定义或本地的工具模块 utils，通常包含项目特定的辅助函数
from opentelemetry import trace  # 从 opentelemetry 核心库导入 trace 接口，用于后续获取 tracer 和管理 span
from opentelemetry.sdk.resources import Resource  # 从 SDK 中导入 Resource 类，用于描述产生追踪数据的实体（如服务名称、版本等）
from opentelemetry.trace import Status, StatusCode  # 导入状态相关类，用于手动标记 span 的执行结果（如 OK、ERROR 等）
from opentelemetry.sdk.trace import TracerProvider  # 导入 TracerProvider，它是追踪 SDK 的核心入口，负责管理和配置追踪流
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor  # 导入控制台导出器和简单处理器，用于将追踪数据直接输出到控制台
import warnings  # 导入 Python 内置的警告管理模块
warnings.filterwarnings("ignore")  # 配置警告过滤器，设置为忽略（不显示）程序运行中的所有警告信息

<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute
<frozen importlib._bootstrap>:241: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute


<a id='2'></a>
## 2 - 遥测技术简要介绍
---
<a id='2-1'></a>
### 2.1 Span (跨度)



在遥测技术中，**Span（跨度）**代表系统内的一个单一操作或任务。它就像一个特定动作的快照，记录了该动作的开始和结束时间。Span 还包含诸如任务正在执行的具体内容以及发生的任何重要事件等详细信息。通过追踪 Span，你可以直观地查看各项操作的耗时并发现潜在问题，从而帮助你理解并优化系统性能。

让我们看一个如何使用 [OpenTelemetry](https://opentelemetry.io/) 设置简单追踪器（tracer）的例子——Phoenix 工具也采用了该技术。

**注意：** 我们在这里创建的是一个**局部追踪器提供者**（而非全局提供者），这使我们能够在不干扰稍后将要使用的 Phoenix 追踪器提供者的情况下，演示 OpenTelemetry 的核心概念。

In [3]:
# 定义一个包含属性的资源对象，用于描述你的应用程序（例如设置服务名称为 "Test Service"）
resource = Resource(attributes={
    "service.name": "Test Service"
})

# 初始化一个本地的追踪提供者（TracerProvider），并将上面定义的资源关联到该提供者上
local_tracer_provider = TracerProvider(resource=resource)

# 创建一个控制台导出器（ConsoleSpanExporter），其作用是将追踪到的 Span 数据直接打印到终端控制台
console_exporter = ConsoleSpanExporter()

# 初始化一个 Span 处理器（SimpleSpanProcessor），它负责在 Span 结束时立即将其交给指定的导出器
span_processor = SimpleSpanProcessor(console_exporter)

# 将定义好的 Span 处理器添加到本地追踪提供者中，使其生效
local_tracer_provider.add_span_processor(span_processor)

# 从该本地提供者中获取一个追踪器（Tracer），通常传入当前模块名 __name__ 以便区分日志来源
tracer = local_tracer_provider.get_tracer(__name__)

#### 2.1.1 一个简单的检索函数 (Toy Retrieve Function)

这是一个基础函数，旨在演示如何为文档检索操作设置基于 span 的追踪。

In [4]:
def retrieve(query, fail=False):  # 定义检索函数，接收查询字符串 query 和一个布尔值 fail（用于模拟失败）
    # 使用 tracer 创建并开启一个名为 "retrieving_documents" 的 Span，并将其作为当前上下文
    with tracer.start_as_current_span("retrieving_documents") as span:
        # 在当前 Span 中添加一个名为 "Starting retrieve" 的事件日志
        span.add_event("Starting retrieve")
        # 为 Span 设置一个自定义属性 "input.query"，记录传入的查询内容
        span.set_attribute("input.query", query)
        try:  # 开启错误捕获代码块
            # 如果参数 fail 为 True，则手动抛出一个 ValueError 异常来模拟检索失败
            if fail:
                raise ValueError(f"Retrieve failed for query: {query}")

            # 模拟从数据库或索引中检索到的文档列表
            retrieved_docs = ['retrieved doc1', 'retrieved doc2', 'retrieved doc3']
            # 遍历检索到的文档列表，记录每个文档的详细信息到 Span 属性中
            for i, doc in enumerate(retrieved_docs):
                # 记录文档的 ID（此处使用循环索引）
                span.set_attribute(f"retrieval.documents.{i}.document.id", i)
                # 记录文档的实际内容
                span.set_attribute(f"retrieval.documents.{i}.document.content", doc)
                # 记录文档的相关元数据
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", f"Metadata for document {i}")
        except Exception as e:  # 捕获执行过程中发生的任何异常
            # 将 Span 的状态设置为 ERROR，并附带异常的描述信息
            span.set_status(Status(StatusCode.ERROR, str(e)))
            # 记录异常的类型名称（如 ValueError）
            span.set_attribute("error.type", type(e).__name__)
            # 记录具体的错误消息内容
            span.set_attribute("error.message", str(e))
            # 重新抛出异常，确保调用者能够感知到错误
            raise

        # 如果代码顺利执行到这里（未发生异常），将 Span 状态标记为 OK
        span.set_status(Status(StatusCode.OK))
        return retrieved_docs  # 返回检索到的文档列表

In [5]:
# 这行注释说明追踪器（Tracer）已经配置完毕，执行结果（Span）将会显示在输出（通常是控制台）中
retrieve("Test")  # 调用前面定义的 retrieve 函数，并将 "Test" 作为查询参数传入，触发整个追踪记录流程

{
    "name": "retrieving_documents",
    "context": {
        "trace_id": "0x3039e082a3cd96a377bd72be22f59659",
        "span_id": "0x0757a173ff78088a",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-21T16:03:45.523996Z",
    "end_time": "2026-04-21T16:03:45.524103Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "input.query": "Test",
        "retrieval.documents.0.document.id": 0,
        "retrieval.documents.0.document.content": "retrieved doc1",
        "retrieval.documents.0.document.metadata": "Metadata for document 0",
        "retrieval.documents.1.document.id": 1,
        "retrieval.documents.1.document.content": "retrieved doc2",
        "retrieval.documents.1.document.metadata": "Metadata for document 1",
        "retrieval.documents.2.document.id": 2,
        "retrieval.documents.2.document.content": "retrieved doc3",
        "retrieval.documents.2.document.metadata": "M

['retrieved doc1', 'retrieved doc2', 'retrieved doc3']

<a id='2-2'></a>
## 2.2 追踪 (Traces)
---
追踪（Trace）是多个 Span 的集合，代表了一个请求或事务在穿过系统中各个组件时的完整旅程。它是一组与同一个任务相关的 Span。



现在让我们完成一个简单的 RAG 流水线，看看追踪（Trace）长什么样。

In [6]:
# 文档格式化  此函数将检索到的列表转换为长字符串，并记录处理每个文档的明细。
def format_documents(retrieved_docs):
    # 启动一个名为 "call_format_documents" 的 Span 来追踪文档格式化过程
    with tracer.start_as_current_span("call_format_documents") as span:
        # 添加一个事件，记录格式化操作正式启动
        span.add_event("Calling format_documents")
        # 将输入文档的数量记录为属性，方便后续分析处理规模
        span.set_attribute("input.documents_count", len(retrieved_docs))

        t = ''  # 初始化一个空字符串用于存储合并后的内容
        for i, doc in enumerate(retrieved_docs):  # 遍历检索到的文档列表
            t += f'Retrieved doc: {doc}\n'  # 将文档内容按行拼接
            # 为每一个被处理的文档添加一个独立事件，并附带该文档的具体内容作为元数据
            span.add_event(f"processed document {i}", {"document.content": doc})

        # 完成格式化逻辑后，手动将 Span 状态设置为 OK（成功）
        span.set_status(Status(StatusCode.OK))
    return t  # 返回合并后的字符串

# Prompt 增强   此函数将查询与检索内容结合。
def augment_prompt(query, formatted_documents):
    # 启动一个名为 "augment_prompt" 的 Span 来追踪 Prompt 构建过程
    with tracer.start_as_current_span("augment_prompt") as span:
        # 记录 Prompt 增强步骤开始的事件
        span.add_event("Starting prompt augmentation")
        # 记录查询内容和格式化后文档的总长度，作为可搜索的属性
        span.set_attribute("input.query", query)
        span.set_attribute("input.formatted_documents_length", len(formatted_documents))

        # 核心逻辑：将用户查询和格式化后的参考文档拼接成最终的 Prompt 模板
        PROMPT = f"Answer the query: {query}.\nRelevant documents:\n{formatted_documents}"

        # 标记当前 Span 状态为成功
        span.set_status(Status(StatusCode.OK))
    return PROMPT  # 返回构建好的 Prompt

# 结果生成  模拟 LLM 生成答案的过程。
def generate(prompt):
    # 启动一个名为 "generate" 的 Span 来追踪文本生成耗时
    with tracer.start_as_current_span("generate") as span:
        # 添加开始生成的事件记录
        span.add_event("Starting text generation")
        # 将发送给模型（模拟）的 Prompt 记录在属性中，便于调试
        span.set_attribute("input.prompt", prompt)

        # 模拟模型生成响应的过程
        generated_text = f"Generated text for prompt {prompt}"

        # 记录生成成功
        span.set_status(Status(StatusCode.OK))
    return generated_text  # 返回生成的文本

# 核心流水线   这是整个流程的“指挥官”，它负责协调上述所有函数。
def rag_pipeline(query, fail = False):
    # 启动顶级父 Span "rag_pipeline"，此 Span 将包含上述所有子操作的 Span
    with tracer.start_as_current_span("rag_pipeline") as span:
        try:  # 开启异常监控块
            # 步骤 1: 根据查询检索文档（调用之前定义的 retrieve 函数）
            retrieved_docs = retrieve(query, fail = fail)
            # 步骤 2: 将检索到的多个文档格式化为统一文本
            formatted_docs = format_documents(retrieved_docs)
            # 步骤 3: 结合查询和背景文档构建增强 Prompt
            prompt = augment_prompt(query, formatted_docs)
            # 步骤 4: 调用模拟模型生成最终回答
            generated_response = generate(prompt)

            # 如果上述所有步骤都成功运行，将父 Span 状态设为 OK
            span.set_status(Status(StatusCode.OK))
            return generated_response  # 返回最终生成的结果
        except Exception as e:  # 捕获流程中任何环节抛出的异常
            # 如果中间任何一步报错（如检索失败），将父 Span 标记为 ERROR 并记录错误详情
            span.set_status(Status(StatusCode.ERROR, str(e)))
            # 重新抛出异常，以便外部调用方可以进行最后的错误处理
            raise

In [7]:
# Trace example 1  # 追踪示例 1：设置一个注释标签，用于区分不同的测试用例
response = rag_pipeline("This is a test query", fail = False)  # 调用 RAG 全流程函数，传入测试查询内容并将 fail 参数设为 False（确保流程正常执行），执行结果将赋值给 response 变量

{
    "name": "retrieving_documents",
    "context": {
        "trace_id": "0x22a4f5a80697d4389aad45740759eabc",
        "span_id": "0xfcc67cef80e47743",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xad51e47294384b34",
    "start_time": "2026-04-21T16:03:53.449188Z",
    "end_time": "2026-04-21T16:03:53.449240Z",
    "status": {
        "status_code": "OK"
    },
    "attributes": {
        "input.query": "This is a test query",
        "retrieval.documents.0.document.id": 0,
        "retrieval.documents.0.document.content": "retrieved doc1",
        "retrieval.documents.0.document.metadata": "Metadata for document 0",
        "retrieval.documents.1.document.id": 1,
        "retrieval.documents.1.document.content": "retrieved doc2",
        "retrieval.documents.1.document.metadata": "Metadata for document 1",
        "retrieval.documents.2.document.id": 2,
        "retrieval.documents.2.document.content": "retrieved doc3",
        "retrieval.do

In [8]:
# Trace example 2  # 追踪示例 2：模拟一个会发生异常的测试场景
response = rag_pipeline("This is a test query", fail = True)  # 调用 RAG 流水线并将 fail 设为 True。这会触发 retrieve 函数内部的报错逻辑，异常会逐层上传，最终导致此行抛出 ValueError。

{
    "name": "retrieving_documents",
    "context": {
        "trace_id": "0xd18fc0c6bdcc2a30c9de1d37f81477cd",
        "span_id": "0xde7a34df7665e173",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xb9c9f612d89dac70",
    "start_time": "2026-04-21T16:03:56.355967Z",
    "end_time": "2026-04-21T16:03:56.359489Z",
    "status": {
        "status_code": "ERROR",
        "description": "ValueError: Retrieve failed for query: This is a test query"
    },
    "attributes": {
        "input.query": "This is a test query",
        "error.type": "ValueError",
        "error.message": "Retrieve failed for query: This is a test query"
    },
    "events": [
        {
            "name": "Starting retrieve",
            "timestamp": "2026-04-21T16:03:56.355981Z",
            "attributes": {}
        },
        {
            "name": "exception",
            "timestamp": "2026-04-21T16:03:56.359468Z",
            "attributes": {
                "exception.t

ValueError: Retrieve failed for query: This is a test query

<div class="alert alert-block alert-warning">
注意：这第二个追踪（trace）是有意设置为失败的，目的是演示在生产系统中这种情况的表现形式。
</div>

如你所见，原始形式的追踪可能会变得非常复杂且难以阅读，特别是在具有许多互连组件的大型系统中。这就是为什么像 **Phoenix** 这样的工具至关重要。它们有助于管理和可视化追踪数据，从而更轻松地分析数据，并高效地诊断性能问题或瓶颈。

<a id='3'></a>
## 3 - 使用 Phoenix 进行遥测
---

Phoenix 是一款功能强大的工具，旨在简化遥测数据的管理和可视化。它能帮助你处理复杂的追踪（traces），让你更轻松地分析和诊断系统中的问题。通过 Phoenix，你可以监控 RAG 系统的性能，识别瓶颈，并深入了解应用程序中不同组件之间的交互过程。在本节中，你将探索如何设置 Phoenix，并利用其功能来更好地理解 RAG 系统生成的遥测数据。

In [9]:
import utils  # 导入自定义的工具模块 utils，通常用于处理项目特定的辅助任务
import phoenix as px  # 导入 Arize Phoenix 库并简写为 px，这是一个用于 AI/LLM 模型观测、评估和追踪的平台

<a id='3-1'></a>
### 3.1 启动 Phoenix 应用

运行下一个单元格来启动 Phoenix 应用。这会建立一个本地服务器并托管一个用户界面 (UI)。访问该应用的默认 URL 是 `localhost:6006`，调用应用程序后会显示该地址。不过，由于 Coursera 环境的限制，我们将提供一个不同的链接。你可以点击该替代链接，在新的标签页中打开 UI 界面。

In [10]:
utils.make_url()  # 调用工具函数生成一个访问 URL，通常用于在特定云端环境（如课程平台）中映射并展示 Phoenix 控制台的链接
px.launch_app()  # 启动 Arize Phoenix 应用程序，它会开启一个本地 Web 服务器，让你可以在浏览器中通过可视化界面查看和分析之前记录的所有 Trace 数据

在本地机器运行 - 使用 localhost
请点击此链接打开 UI 界面: http://localhost:6006


boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix


In [23]:
px.close_app()

你应该看到这样的内容：

![Phoenix UI Screenshot](images/ui_1.png)

<a id='3-2'></a>
### 3.2 准备遥测

现在你将配置遥测以配合 Phoenix 工作。由于 Phoenix 同样使用 OpenTelemetry，其设置过程与你之前看到的非常相似。

In [ ]:
# 不要用这个 因为文档的phoenix版本较低 用下面的

# 旧的：from phoenix.otel import register
# 新的：
from phoenix.trace.otel import register # 从 phoenix.otel 模块导入 register 函数，用于快速配置 OpenTelemetry
from opentelemetry.trace import Status, StatusCode  # 再次导入状态相关类，确保后续手动标记 Span 状态时可用
phoenix_project_name = "example-rag-pipeline"  # 定义在 Phoenix 界面中显示的项目的名称

# 使用 Phoenix 提供的注册功能，只需指定项目名和后端地址即可获取配置好的追踪提供者
endpoint="http://127.0.0.1:6006/v1/traces"  # 设置 Phoenix 接收追踪数据（OTLP）的本地 API 端点地址
tracer_provider_phoenix = register(project_name=phoenix_project_name, endpoint = endpoint)  # 执行注册，返回一个已经连接到 Phoenix 端的 TracerProvider

# 获取一个用于手动埋点的追踪器（Tracer）
tracer = tracer_provider_phoenix.get_tracer(__name__)  # 从 Phoenix 提供者中获取追踪器，并以当前模块名命名

In [15]:
import phoenix as px
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.resources import Resource
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export import BatchSpanProcessor

# 1. 确保 Phoenix 已经启动（你之前已经跑通了）
px.launch_app()

# 2. 定义项目“身份标签” (替代原来的 project_name 参数)
resource = Resource(attributes={"project_name": "example-rag-pipeline"})

# 3. 配置数据“邮差”：将数据发往 Phoenix 默认监听的 6006 端口
# v14 的标准 OTLP 接收路径是 /v1/traces
exporter = OTLPSpanExporter(endpoint="http://127.0.0.1:6006/v1/traces")
processor = BatchSpanProcessor(exporter)

# 4. 创建并配置 Provider (这就是你之前想要的 tracer_provider_phoenix)
tracer_provider_phoenix = TracerProvider(resource=resource)
tracer_provider_phoenix.add_span_processor(processor)

# 5. 获取追踪器 (Tracer)，现在你的 tracer.start_as_current_span 就能用了
tracer = tracer_provider_phoenix.get_tracer(__name__)

from opentelemetry.trace import Status, StatusCode
print("✅ v14 标准追踪模式配置成功！")

Existing running Phoenix instance detected! Shutting it down and starting a new instance...
boto3 is installed but aioboto3 is not. To use AWS Bedrock models in Playground, install aioboto3: pip install aioboto3


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
✅ v14 标准追踪模式配置成功！


<a id='3-3'></a>
### 3.3 使用流水线

#### 3.3.1 检索 (Retrieve)

这是同一个简单的检索函数。语法几乎完全一致，但有两点不同：

1. 引入了 `openinference_span_kind`，你可以通过它将其指定为检索器（retriever）。
2. 你现在可以通过 `span.set_input` 来设置输入。

In [16]:
def retrieve(query, fail=False):  # 定义检索函数，接收查询字符串和失败模拟开关
    # 启动一个名为 "retrieving_documents" 的 Span，并指定 span_kind 为 'retriever' (OpenInference 规范)
    # 这能让 Phoenix 将此步骤识别为“检索器”，并在 UI 中使用专门的图标和布局展示
    with tracer.start_as_current_span("retrieving_documents", openinference_span_kind = 'retriever') as span:
        # 在追踪记录中添加一个名为 "Starting retrieve" 的即时事件
        span.add_event("Starting retrieve")
        # 使用 Phoenix 扩展的 set_input 方法直接记录输入参数
        # 相比普通的 set_attribute，这在 Phoenix 界面中会直接显示在 "Input" 栏目下
        span.set_input(query)
        try:  # 开始业务逻辑监控
            # 如果 fail 为 True，则抛出异常模拟检索环节出错
            if fail:
                raise ValueError(f"Retrieve failed for query: {query}")

            # 模拟获取到的文档列表
            retrieved_docs = ['retrieved doc1', 'retrieved doc2', 'retrieved doc3']
            # 遍历文档，按照 OpenInference 规范记录每个文档的详细属性
            for i, doc in enumerate(retrieved_docs):
                # 记录文档的唯一标识符 ID
                span.set_attribute(f"retrieval.documents.{i}.document.id", i)
                # 记录文档的实际文本内容
                span.set_attribute(f"retrieval.documents.{i}.document.content", doc)
                # 记录文档的元数据信息
                span.set_attribute(f"retrieval.documents.{i}.document.metadata", f"Metadata for document {i}")
        except Exception as e:  # 捕获执行过程中的异常
            # 将 Span 状态标记为 ERROR，并记录异常简述
            span.set_status(Status(StatusCode.ERROR, str(e)))
            # 记录异常的具体类型（如 ValueError）
            span.set_attribute("error.type", type(e).__name__)
            # 记录完整的错误消息
            span.set_attribute("error.message", str(e))
            # 向上层抛出异常，确保持续集成或逻辑处理能感知到错误
            raise

        # 如果运行顺利，将 Span 状态显式设置为 OK
        span.set_status(Status(StatusCode.OK))
        return retrieved_docs  # 返回模拟的文档列表

<a id='3-4'></a>
### 3.4 链 (Chains)

**链 (Chain)** 是 LLM 应用程序中不同步骤之间的连接点。它将各种操作连接在一起，例如启动请求或将信息从检索器传递到 LLM 调用。链有助于保持流程井然有序且简洁。



#### 3.4.1 剩余的 RAG 函数

这些函数与你之前使用的相同，但现在使用了**装饰器** `@tracer.chain`。你只需要将其添加在想要追踪的函数之前，它就会作为一个“链”被添加进去！如果你想进行更详细的追踪（例如手动控制 span），则应按照上面检索器的示例进行操作。

In [21]:
from functools import wraps

def phoenix_trace(span_name):
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            # 这里的 tracer 就是我们之前配置好的专家模式对象
            with tracer.start_as_current_span(span_name) as span:
                return func(*args, **kwargs)
        return wrapper
    return decorator

# @tracer.chain 版本太老  # 使用装饰器自动追踪此函数，Phoenix 会将其记录为一个名为 "format_documents" 的 Span
@phoenix_trace("format_documents")  # 使用这个代替 tracer.chain
def format_documents(retrieved_docs):  # 定义文档格式化函数，接收检索到的文档列表
    t = ''  # 初始化一个空字符串，用于累加格式化后的文本
    for i, doc in enumerate(retrieved_docs):  # 遍历文档列表及其索引
        t += f'Retrieved doc: {doc}\n'  # 将每个文档内容拼接成带标签的行
    return t  # 返回拼接后的完整字符串，装饰器会自动将此返回值记录为 Span 的 Output

# @tracer.chain  # 自动追踪此函数，记录 Prompt 增强的过程
@phoenix_trace("augment_prompt")
def augment_prompt(query, formatted_documents):  # 定义增强函数，接收原始查询和格式化后的文档
    
    # 将用户查询与参考文档组合，构建成最终发送给 LLM 的提示词模板
    PROMPT = f"Answer the query: {query}.\nRelevant documents:\n{formatted_documents}"
    return PROMPT  # 返回构建好的 Prompt，装饰器会自动捕获该字符串

# @tracer.chain  # 自动追踪此函数，模拟生成环节
@phoenix_trace("generate")
def generate(prompt):  # 定义生成函数，接收 Prompt
    generated_text = f"Generated text for prompt {prompt}"  # 模拟模型根据 Prompt 生成响应内容
    return generated_text  # 返回生成结果

# @tracer.chain  # 自动追踪顶级流水线，它将作为父 Span 包含下面所有子函数的调用记录
@phoenix_trace("rag_pipeline")
def rag_pipeline(query, fail = False):  # 定义 RAG 流水线主函数
        # 步骤 1: 调用之前定义的 retrieve 函数进行文档检索（手动埋点版本）
        retrieved_docs = retrieve(query, fail = fail)
        # 步骤 2: 格式化检索到的文档（由 @tracer.chain 自动记录）
        formatted_docs = format_documents(retrieved_docs)
        # 步骤 3: 构建增强后的 Prompt（由 @tracer.chain 自动记录）
        prompt = augment_prompt(query, formatted_docs)
        # 步骤 4: 基于 Prompt 生成最终回答（由 @tracer.chain 自动记录）
        generated_response = generate(prompt)
        return generated_response  # 返回最终生成的答案

<a id='3-5'></a>
### 3.5 使用 UI 分析追踪数据

现在到了有趣的部分！运行下面的单元格来执行与之前相同的两个查询。然后，让我们前往 Phoenix UI 查看它们是如何显示的！

In [22]:
# 执行第一个测试用例：正常调用 RAG 流水线并获取返回结果
response = rag_pipeline("This is a test query")

# 开启一个异常处理块，用于捕获第二个测试用例中预期的异常
try:
    # 执行第二个测试用例：将 fail 参数设为 True，这将触发内部 retrieve 函数的报错逻辑
    response = rag_pipeline("This is a test query that failed", fail = True)
# 捕获所有类型的异常（在这里主要是为了防止程序因为预期的错误而崩溃）
except:
    # 捕获异常后不进行任何操作，直接跳过，确保后续代码（或脚本运行）能继续进行
    pass

TypeError: Tracer.start_as_current_span() got an unexpected keyword argument 'openinference_span_kind'

In [ ]:
utils.make_url()  # 调用工具函数生成一个可访问的外部链接，主要用于在云端环境中将本地运行的 Phoenix 界面映射到公网，方便你点击查看可视化追踪看板。

你会看到类似这样的东西，所有的信息都以一种有组织的方式显示给你。


![Phoenix UI Screenshot](images/ui_3.png)

In [ ]:
# Restart kernel to demonstrate a fresh Phoenix setup with auto_instrument enabled
# 重启内核以演示在开启 auto_instrument（自动插桩）情况下的全新 Phoenix 配置

# This restart allows us to show two different Phoenix configurations:
# 此次重启允许我们展示两种不同的 Phoenix 配置：

# - Previous section: Basic Phoenix tracing
# - 前一部分：基础的 Phoenix 追踪（手动埋点）

# - Next section: Phoenix with auto_instrument=True for automatic OpenAI tracing
# - 下一部分：使用 auto_instrument=True 来实现针对 OpenAI 的全自动追踪

# Note: In production, you would typically configure Phoenix once at application startup
# 注意：在生产环境中，你通常只需要在应用程序启动时配置一次 Phoenix 即可

utils.restart_kernel()  # 调用工具函数重启 Python 内核，这会清空内存中的所有变量、已导入的模块以及之前配置的全局追踪器（Tracer）

<a id='4'></a>
## 4 - 结合 Weaviate 的追踪与评估

---

既然你已经熟悉了使用 Phoenix 进行遥测的基础知识，让我们来看一个更具体的场景。我们将获取来自 M4 作业的 FAQ（常见问题）数据，并实现一个小型的 RAG 流水线，以回答一家服装店的相关问题。

In [ ]:
from phoenix.otel import register  # 从 phoenix.otel 导入注册函数，用于配置 OpenTelemetry 数据导出
from opentelemetry.trace import Status, StatusCode  # 导入 OTel 的状态追踪类，用于后续手动标记 Span 成功或失败
import phoenix as px  # 导入 Arize Phoenix 库，简写为 px
import utils  # 导入本地工具模块，用于执行环境清理和内核操作
import weaviate  # 导入 Weaviate 客户端库，这是一个用于存储和检索向量数据的数据库

# 在启动新服务之前，强制关闭占用特定端口的旧进程，确保环境纯净
# 这些端口通常对应：5000 (Flask), 8080 (Weaviate REST API), 50051 (Weaviate gRPC) 等
utils.kill_processes_on_ports([5000, 8080, 8097, 50050, 50051])

import flask_app  # 导入并运行本地的 Flask 应用模块，通常作为 RAG 系统的 API 入口
import weaviate_server  # 导入并启动本地的 Weaviate 服务器模块，准备处理向量搜索请求

In [ ]:
# 调用工具函数清理所有现有的 Phoenix 项目，以解决由于重复使用项目 ID 可能导致的冲突或数据混淆
utils.cleanup_phoenix_projects()

# 再次调用此工具函数，确保在当前环境下生成并显示正确的 Phoenix 控制台访问 URL
utils.make_url()

# 启动 Phoenix 应用程序，并将返回的会话对象赋值给变量 session，以便后续对该会话进行操作或状态查询
session = px.launch_app()

<a id='4-1'></a>
### 4.1 配置追踪器

设置过程与之前相同，但现在有一个名为 `auto_instrument` 的新参数。将其设置为 `True` 将会自动追踪与 OpenAI 兼容的 LLM 调用！

In [ ]:
from phoenix.otel import register  # 从 phoenix.otel 模块导入 register 函数，用于快速配置 OpenTelemetry 导出逻辑
import time  # 导入内置 time 模块，用于生成时间戳

# 使用当前时间戳生成一个唯一的项目名称，防止多次运行代码时在 Phoenix 中产生命名冲突
phoenix_project_name = f"example-rag-pipeline-with-weaviate-{int(time.time())}"

# 使用 register 函数配置追踪提供者。
# 关键点：设置 auto_instrument=True 后，系统会自动拦截并追踪所有 OpenAI SDK 的调用（包括兼容 OpenAI 接口的 TogetherAI）
tracer_provider_phoenix = register(
    project_name=phoenix_project_name, 
    endpoint="http://127.0.0.1:6006/v1/traces", 
    auto_instrument=True
)

# 从配置好的 Phoenix 提供者中获取一个追踪器（Tracer），以便在自动追踪的基础上进行必要的手动埋点
tracer = tracer_provider_phoenix.get_tracer(__name__)

<a id='4-2'></a>
### 准备Weaviate客户端和集合


In [ ]:
# 使用 Weaviate 客户端连接到本地运行的数据库实例
# port=8079 用于标准的 RESTful API 通信
# grpc_port=50050 用于高性能的数据检索和流式传输（Weaviate v4+ 的核心特性）
client = weaviate.connect_to_local(port=8079, grpc_port=50050)

# 调用工具函数初始化 FAQ 集合
# 该步骤通常包括：创建 Schema（架构）、配置向量索引、并将预设的问答对数据导入数据库
utils.setup_faq_collection()

In [ ]:
import joblib  # 导入 joblib 库，它通常用于高效地序列化和反序列化包含大型数据（如 NumPy 数组）的 Python 对象
data = joblib.load("faq.joblib")  # 使用 joblib 的 load 方法从磁盘读取 "faq.joblib" 文件，并将其内容还原为 Python 对象赋值给 data 变量

In [ ]:
# Let's recall the data structure
data[0]

In [ ]:
# Loading the collection  # 加载集合：这是一个注释，说明接下来的操作是连接到特定的数据表
collection = client.collections.get("Faq")  # 使用 Weaviate 客户端对象的 collections.get 方法获取名为 "Faq" 的集合（Collection）对象。之后所有的查询、插入和删除操作都将通过这个 collection 变量进行。

In [ ]:
len(collection)

<a id='4-3'></a>
### 4.3 检索器 (The Retriever)

现在你将像之前一样设置一个检索器。这一次，你还将添加遥测技术，以追踪并理解检索过程。

In [ ]:
def retrieve(query_text, limit=5):  # 定义检索函数，接收查询文本和返回数量限制（默认为 5）
    # 启动一个名为 "query_weaviate" 的 Span，并标记其类型为 "retriever"（检索器）
    with tracer.start_as_current_span(
        "query_weaviate", openinference_span_kind="retriever"
    ) as span:
        # 在 Phoenix/OpenTelemetry 中记录该步骤的输入查询字符串
        span.set_input(query_text)

        # 指定要查询的集合名称为 "Faq"
        collection_name = "Faq"
        # 从 Weaviate 客户端获取该集合的操作对象
        chunks = client.collections.get(collection_name)
        # 执行近义词/向量搜索，根据输入文本检索最相关的文档对象
        results = chunks.query.near_text(query=query_text, limit=limit)

        # 遍历检索到的结果对象列表
        for i, document in enumerate(results.objects):
            # 记录每个文档的唯一标识符 UUID 
            span.set_attribute(f"retrieval.documents.{i}.document.id", str(document.uuid))
            # 记录该文档的元数据（如创建时间、作者等）
            span.set_attribute(f"retrieval.documents.{i}.document.metadata", str(document.metadata))
            # 记录该文档的实际内容属性（即数据库中存储的键值对内容）
            span.set_attribute(
                f"retrieval.documents.{i}.document.content", str(document.properties)
            )  

        return results  # 返回包含检索结果的对象

In [ ]:
# 处理并格式化检索到的结果（注释说明函数用途）
@tracer.chain  # 使用 Phoenix 的装饰器自动追踪此函数。它会自动记录输入（results）和输出（context），并在 Trace 视图中生成一个子 Span
def format_context(results):  # 定义名为 format_context 的函数，接收 Weaviate 的检索结果对象
    context = ""  # 初始化一个空字符串，用于累加存储所有的问答对文本
    for item in results.objects:  # 遍历 Weaviate 检索结果中的每一个数据对象
        properties = item.properties  # 获取该对象的属性字典（包含数据库中存储的实际字段）
        context += f"Question: {properties['question']}\n"  # 从属性中提取“问题”字段，格式化后拼接到 context 字符串中
        context += f"Answer: {properties['answer']}\n"  # 从属性中提取“答案”字段，格式化后拼接到 context 字符串中
    return context  # 返回拼接完成的完整上下文字符串，装饰器会自动将其捕获为该步骤的 Output

In [ ]:
# 使用检索到的信息创建一个 Prompt（注释说明函数用途）
@tracer.chain  # 使用装饰器自动追踪此函数。它会自动记录输入的 query_text 和 context，并将生成的 prompt 记录为该 Span 的输出
def create_prompt(query_text, context):  # 定义函数，接收用户原始查询和格式化后的上下文字符串
    # 使用 Python 的多行字符串（f-string）构建发送给 LLM 的最终提示词
    prompt = f"""
Based on the following information, please answer the FAQ related question: "{query_text}"

Relevant FAQ (ordered by relevance):
{context}
"""
    return prompt  # 返回构建好的完整提示词字符串

<a id='4-4'></a>
### 4.4 使用 `openai` 库进行 LLM 调用

由于 Phoenix 与类 OpenAI 系统具有良好的集成性，让我们开始使用它。幸运的是，[together.ai](https://www.together.ai/) 与 OpenAI 协议是兼容的！

In [ ]:
import httpx  # 导入 httpx 库，这是一个支持同步和异步请求的现代化 HTTP 客户端，功能比传统的 requests 更强大
from openai import OpenAI, DefaultHttpxClient  # 导入 OpenAI 客户端主类，以及默认的 HTTP 客户端组件（用于后续自定义网络配置）

In [ ]:
# 创建自定义的 HTTP 传输层，local_address 指定本地绑定地址，verify=False 表示跳过 SSL 证书验证（用于绕过代理产生的证书问题）
transport = httpx.HTTPTransport(local_address="0.0.0.0", verify=False)

# 使用上面定义的自定义传输层创建一个 HTTP 客户端实例，并从 utils 模块中获取必要的代理请求头
http_client = DefaultHttpxClient(transport=transport, headers=utils.get_proxy_headers())

# 初始化 OpenAI 客户端对象（此处可兼容任何支持 OpenAI 协议的端点，如 TogetherAI）
llm_client = OpenAI(
    api_key = utils.get_together_key(), # 从工具类获取 API 密钥（如果通过代理转发，有时该值可为任意字符串，但使用 TogetherAI 时需真实密钥）
    base_url = utils.get_proxy_url(), # 设置 API 的基础地址，utils 会根据当前环境（如本地或云端）自动检测正确的代理 URL
    http_client = http_client, # 将之前配置好的、跳过 SSL 验证的自定义客户端注入到 OpenAI SDK 中
)

In [ ]:
# 由于之前设置了 auto_instrument = True，此处无需手动编写追踪代码，OpenAI 库的调用会被自动捕获
def query_openai(prompt):  # 定义函数，接收最终生成的提示词字符串
    response = llm_client.chat.completions.create(  # 调用 LLM 的聊天补全接口。此行执行时，Phoenix 会自动记录输入、输出及 Token 消耗
        model="Qwen/Qwen3.5-9B",  # 指定使用的模型 ID（这里使用的是 Qwen 系列模型）
        extra_body={"reasoning": False},  # 向 API 传递额外参数，此处设置为 False 以禁用某些模型可能带有的推理（Thought）过程输出
        messages=[  # 定义对话上下文列表
            {"role": "system", "content": "You are a helpful assistant from a customer support."},  # 设置系统指令，规定 AI 以“客服助手”的身份进行回复
            {"role": "user", "content": prompt},  # 将我们之前构建好的、包含上下文和问题的 prompt 作为用户输入发送
        ],
    )
    return response.choices[0].message.content  # 从复杂的响应对象中提取出模型生成的纯文本回答并返回

In [ ]:
@tracer.chain  # 自动追踪此函数。它将作为父 Span，记录整个 RAG 流水线的总耗时、输入（query）和最终输出（final_answer）
def rag_pipeline(query):  # 定义名为 rag_pipeline 的主函数，接收用户的原始查询
    # Execute the query  # 注释：执行查询逻辑
    retrieved_documents = retrieve(query)  # 步骤 1：调用检索函数，从 Weaviate 向量数据库中查找相关的文档对象
    
    # 步骤 2：调用格式化函数，将数据库返回的对象列表转换为 LLM 可读的“问题-答案”上下文字符串
    context = format_context(retrieved_documents)
    
    # Create a prompt with the retrieved information  # 注释：使用检索到的信息构建提示词
    # 步骤 3：将用户原始问题和刚生成的上下文填入预设模板，合成最终的提示词
    final_prompt = create_prompt(query, context)
    
    # Execute the OpenAI query  # 注释：执行 OpenAI 查询
    # 步骤 4：将合成的提示词发送给大模型。注意：此处的追踪是由之前配置的 auto_instrument 自动完成的
    final_answer = query_openai(final_prompt)

    return final_answer  # 返回大模型生成的最终答案，该值会被装饰器记录为流水线的最终输出

In [ ]:
# 调用整个 RAG 流水线主函数，传入关于“退款或换货”的客服咨询问题
# 该操作会依次触发：向量检索 -> 上下文格式化 -> Prompt 构建 -> LLM 生成回答
response = rag_pipeline("Can I get a refund or exchange for another shirt?")

# 在控制台打印出大模型根据检索到的 FAQ 知识所生成的最终回答
print(response)

In [ ]:
response = rag_pipeline("What are your working hours?")
print(response)

In [ ]:
# Checkout the traces in the Phoenix UI!  # 这是一个提示性的注释，提醒你去 Phoenix 的 Web 界面中查看刚才 RAG 流水线运行产生的详细追踪数据（Traces）
utils.make_url()  # 调用工具函数生成并打印出访问 Phoenix UI 的链接。在云端开发环境中，这会为你提供一个可以直接点击跳转的可视化看板入口。

Keep it up! You finished the ungraded lab on Telemetry with Phoenix!